# FlyLingo: does a real fly brain's wiring matter, on a task it can learn?

This notebook runs one controlled experiment. A fruit-fly connectome — the
[Janelia/Google MaleCNS v1.0](https://www.janelia.org/project-team/flyem), 166,700 neurons and
25,582,938 directed edges — is used as a reservoir that answers multiple-choice Spanish questions,
and the answer is read out of the activity of its own neurons. Four versions of the brain are
trained identically and compared:

| arm | what the wiring is |
|---|---|
| `intact` | the measured connectome |
| `shuffled` | the same graph, nodes relabelled (topology kept, interfaces moved) |
| `random_graph` | a degree-matched random matrix with the same number of edges |
| `no_edges` | disconnected — the control that must fail |

**What this establishes:** whether the specific measured wiring gives any advantage over a
same-sized graph that is not the fly's.

**What it does NOT establish, stated here rather than in a footnote:**

- This is **memorisation of a 97-item phrase-to-answer mapping**, not language. Every phrase is
  shown every epoch. Nothing here generalises to unseen sentences.
- The connectome is used as a **reservoir**, not as a simulation of a fly. No claim is made that
  the fly's brain "speaks Spanish", or that these dynamics resemble anything biological.
- The answer pools are **disjoint groups of real neurons chosen by a seeded shuffle**. They are not
  identified cell types and are not claimed to be.

**Runtime:** on a free Colab GPU, roughly 25 minutes at the default budget below.

## 1. Environment

Colab already ships `cupy`, `pyarrow` and `scipy`. This checks the GPU and installs only what is missing.

In [ ]:
import subprocess, sys, json, hashlib, base64, os, time
from pathlib import Path

def sh(cmd):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)

need = []
for mod, pkg in (("pyarrow", "pyarrow"), ("scipy", "scipy")):
    try:
        __import__(mod)
    except ImportError:
        need.append(pkg)
try:
    import cupy
    print("cupy", cupy.__version__)
except Exception:
    need.append("cupy-cuda12x")
if need:
    sh([sys.executable, "-m", "pip", "install", "-q", *need])

import cupy
props = cupy.cuda.runtime.getDeviceProperties(0)
print("GPU:", props["name"].decode())
print("All statements below are produced by the code in this notebook.")

## 2. The code, embedded

This repository has no public remote, so the notebook cannot clone it. The three tested modules are
embedded as base64 and written to disk with their SHA-256 verified as they land, so the notebook is
self-contained and the files are provably the ones the project's own test suite covers.

They are **not** reimplemented here. `step1_build.py` builds the connectome and gates the
accelerator; `step2_measure.py` runs the four-arm comparison.

In [ ]:
FILES = {
    'payload.py': '...',
    'step1_build.py': '...',
    'step2_measure.py': 'IiIiQ29sYWIgc3RlcCAyOiB0aGUgZm91ci1hcm0gbWVhc3VyZW1lbnQsIHdpdGggdGhlIGFydGlmYWN0cyB0aGF0IG1ha2UgaXQgY2hlY2thYmxlLgoKVGhlIHF1ZXN0aW9uOiBkb2VzIHRoZSByZWFsIGNvbm5lY3RvbWUncyB3aXJpbmcgYmVhdCBpdHMgY29udHJvbHMgb25jZSB0aGUgYnJhaW4gZG9lcyBib3RoCnRoZSBjaG9vc2luZyBhbmQgdGhlIGxlYXJuaW5nPwoKICAgIGludGFjdCAgICAgICAgdGhlIG1lYXN1cmVkIE1hbGVDTlMgY29ubmVjdG9tZQogICAgc2h1ZmZsZWQgICAgICBzYW1lIGdyYXBoIHVuZGVyIGEgZml4ZWQgbm9kZSByZWxhYmVsaW5nICh0b3BvbG9neSBrZXB0LCBpbnRlcmZhY2VzIG1vdmVkKQogICAgcmFuZG9tX2dyYXBoICBkZWdyZWUtbWF0Y2hlZCByYW5kb20gc3BhcnNlIG1hdHJpeCwgc2FtZSBubnoKICAgIG5vX2VkZ2VzICAgICAgZGlzY29ubmVjdGVkOyBzdGF0ZSBpcyBleGFjdGx5IHplcm8sIHNvIHRoZSBhcmdtYXggaXMgZGVnZW5lcmF0ZQoKRXZlcnkgYXJtIGdldHMgdGhlIHNhbWUgcG9vbHMsIHRoZSBzYW1lIHNldHRsaW5nLCB0aGUgc2FtZSBwbGFzdGljIGVkZ2Ugc2V0LCB0aGUgc2FtZQpleGFtcGxlIG9yZGVyIGFuZCBpdHMgb3duIHByaXN0aW5lIGNvcHkgb2YgdGhlIHdlaWdodHMuIE9ubHkgdGhlIHdpcmluZyBkaWZmZXJzLgoKUnVsZXMgdGhpcyBzY3JpcHQgb2JleXMsIGVhY2ggb2Ygd2hpY2ggd2FzIGxlYXJuZWQgdGhlIGhhcmQgd2F5IG9uIHRoaXMgcHJvamVjdDoKCiAgKiBBbiBhcm0ncyBzY29yZSBpcyB0aGUgbWVhbiBvdmVyIHRoZSBsYXN0IFRBSUwgZXBvY2hzLCBuZXZlciBhIHNpbmdsZSBlcG9jaC4gQSBzaW5nbGUKICAgIGVwb2NoIHN3aW5ncyBieSBzZXZlcmFsIHBvaW50cywgc28gb25lIGRyYXcgaXMgbm90IGEgbWVhc3VyZW1lbnQuCiAgKiBTZXZlcmFsIHNlZWRzLCBjb21wYXJlZCBQQUlSRUQgcGVyIHNlZWQsIHNvIGEgc2VlZCdzIGRpZmZpY3VsdHkgY2FuY2VscyBhbmQgb25seSB0aGUKICAgIG1lY2hhbmlzbSB1bmRlciB0ZXN0IGRpZmZlcnMuCiAgKiBFdmVyeSBhcm0gc2F2ZXMgaXRzIHRyYWluZWQgd2VpZ2h0cywgYW5kIHRoZSByZXBvcnRlZCBudW1iZXIgaXMgcmVjb21wdXRlZCBieSBsb2FkaW5nCiAgICB0aGF0IGNoZWNrcG9pbnQgYmFjayB0aHJvdWdoIHRoZSBzYW1lIGd1YXJkcyB0aGUgc2VydmljZSB1c2VzLiBBIG1pc21hdGNoIGZhaWxzIHRoZSBhcm0uCiAgKiBObyBwcmUtd3JpdHRlbiB2ZXJkaWN0LiBUaGUgc3VtbWFyeSBwcmludHMgdGhlIGVmZmVjdCBuZXh0IHRvIGl0cyBvd24gc3ByZWFkIGFuZAogICAgcmVmdXNlcyB0byBjbGFpbSB3aGVuIHRoZSBlZmZlY3QgZG9lcyBub3QgY2xlYXIgaXQuCiAgKiBQcm9ncmVzcyBpcyBwcmludGVkIHBlciBlcG9jaCBzbyBhIHN0YWxsIGlzIGRpc3Rpbmd1aXNoYWJsZSBmcm9tIHByb2dyZXNzIGluIHRoZSBsb2cuCgpVc2FnZTogIHB5dGhvbiBzdGVwMl9tZWFzdXJlLnB5IFtlcG9jaHNdIFtzZWVkc10gW3RhaWxdCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN0YXRpc3RpY3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQppbXBvcnQgemlwZmlsZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAoKIyBPbmUgZmlsZSwgYm90aCBsYXlvdXRzLiBST09UIGlzIHRoZSBkaXJlY3RvcnkgdGhhdCBjb250YWlucyB0aGUgYGJyYWluYCBwYWNrYWdlIChhbmQKIyBicmFpbi9jdXJyaWN1bHVtKTsgR1JBUEggaXMgdGhlIGJ1aWx0IGNvbm5lY3RvbWUncyBkaXJlY3RvcnksIHdoaWNoIGlzIE5PVCB1bmRlciBST09UIGxvY2FsbHkuCiMgVGhlIGlkZW50aWNhbCBzY3JpcHQgdGhlcmVmb3JlIHJ1bnMgb24gQ29sYWIgYW5kIG9uIHRoaXMgbWFjaGluZS4KUk9PVCA9IFBhdGgob3MuZW52aXJvbi5nZXQoIkZMWUxJTkdPX1JPT1QiLCAiL2NvbnRlbnQvZmx5bGluZ28iKSkKR1JBUEggPSBQYXRoKG9zLmVudmlyb24uZ2V0KCJGTFlMSU5HT19HUkFQSCIsIHN0cihST09UIC8gImNhY2hlIiAvICJtYWxlY25zX3YxIikpKQpSVU5TID0gUGF0aChvcy5lbnZpcm9uLmdldCgiRkxZTElOR09fUlVOUyIsIHN0cihST09UIC8gInJ1bnMiIC8gInBsYXN0aWNfYnJhaW4iKSkpClVTRV9HUFUgPSBvcy5lbnZpcm9uLmdldCgiRkxZTElOR09fR1BVIiwgIjEiKSBub3QgaW4gKCIwIiwgImZhbHNlIiwgIm5vIikKUlVOUy5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCgpFUE9DSFMgPSBpbnQoc3lzLmFyZ3ZbMV0pIGlmIGxlbihzeXMuYXJndikgPiAxIGVsc2UgNDAKU0VFRFMgPSBpbnQoc3lzLmFyZ3ZbMl0pIGlmIGxlbihzeXMuYXJndikgPiAyIGVsc2UgNQpUQUlMID0gaW50KHN5cy5hcmd2WzNdKSBpZiBsZW4oc3lzLmFyZ3YpID4gMyBlbHNlIDEwCk1PREVTID0gKCJpbnRhY3QiLCAic2h1ZmZsZWQiLCAicmFuZG9tX2dyYXBoIiwgIm5vX2VkZ2VzIikKCnN5cy5wYXRoLmluc2VydCgwLCBzdHIoUk9PVCkpCmZyb20gYnJhaW4uZW5jb2RlcnMgaW1wb3J0IGVuY29kZV90ZXh0ICAjIG5vcWE6IEU0MDIKZnJvbSBicmFpbi5wbGFzdGljX2JyYWluIGltcG9ydCBQbGFzdGljQnJhaW4gICMgbm9xYTogRTQwMgpmcm9tIGJyYWluLnJlc2Vydm9pciBpbXBvcnQgRmx5UmVzZXJ2b2lyLCBsb2FkX2Nvbm5lY3RvbWUgICMgbm9xYTogRTQwMgoKY3VyID0ganNvbi5sb2FkKG9wZW4oUk9PVCAvICJicmFpbiIgLyAiY3VycmljdWx1bSIgLyAiZXMtZW4uanNvbiIsIGVuY29kaW5nPSJ1dGYtOCIpKQppdGVtcyA9IFsKICAgIChjWyJwcm9tcHQiXSwgaW50KGNbImNvcnJlY3RJbmRleCJdKSkKICAgIGZvciB1IGluIGN1clsidW5pdHMiXQogICAgZm9yIGwgaW4gdVsibGVzc29ucyJdCiAgICBmb3IgYyBpbiBsWyJjaGFsbGVuZ2VzIl0KXQpYID0gbnAuc3RhY2soW2VuY29kZV90ZXh0KHApIGZvciBwLCBfIGluIGl0ZW1zXSkKeSA9IG5wLmFycmF5KFt0IGZvciBfLCB0IGluIGl0ZW1zXSwgZHR5cGU9aW50KQpjaGFuY2UgPSBmbG9hdChucC5iaW5jb3VudCh5KS5tYXgoKSAvIGxlbih5KSkKCnByaW50KGYidGFzazoge2xlbihpdGVtcyl9IGNoYWxsZW5nZXMsIHtsZW4obnAudW5pcXVlKHkpKX0gY2xhc3NlcywgIgogICAgICBmImFsbCBzaG93biBldmVyeSBlcG9jaCIsIGZsdXNoPVRydWUpCnByaW50KGYibWVtb3Jpc2F0aW9uIG9mIGEgcGhyYXNlLXRvLWFuc3dlciBtYXBwaW5nLCBub3QgbGFuZ3VhZ2UuIiwgZmx1c2g9VHJ1ZSkKcHJpbnQoZiJtYWpvcml0eS1jbGFzcyBiYXNlbGluZSB7Y2hhbmNlOi4xJX0gICB1bmlmb3JtIGNoYW5jZSAyNS4wJSIsIGZsdXNoPVRydWUpCnByaW50KGYiZXBvY2hzIHtFUE9DSFN9ICAgc2VlZHMge1NFRURTfSAgIHNjb3JlID0gbWVhbiBvZiBsYXN0IHtUQUlMfSBlcG9jaHMiLCBmbHVzaD1UcnVlKQoKY29ubmVjdG9tZSA9IGxvYWRfY29ubmVjdG9tZShzdHIoR1JBUEgpKQpwcmludChmImNvbm5lY3RvbWU6IHtjb25uZWN0b21lLm5ldXJvbnN9IG5ldXJvbnMsIHtjb25uZWN0b21lLmVkZ2VzfSBkaXJlY3RlZCBlZGdlcyIsCiAgICAgIGZsdXNoPVRydWUpCgpwZXJfc2VlZDogZGljdFtpbnQsIGRpY3RdID0ge30KZm9yIHNlZWRfaSBpbiByYW5nZShTRUVEUyk6CiAgICByX3NlZWQsIGJfc2VlZCA9IDczMDEgKyBzZWVkX2ksIDk5ICsgc2VlZF9pCiAgICBwcmludChmIlxueycjJyAqIDY4fVxuIyBzZWVkIHtzZWVkX2kgKyAxfS97U0VFRFN9ICAocmVzZXJ2b2lyIHtyX3NlZWR9LCBicmFpbiB7Yl9zZWVkfSkiCiAgICAgICAgICBmIlxueycjJyAqIDY4fSIsIGZsdXNoPVRydWUpCiAgICBwZXJfc2VlZFtzZWVkX2ldID0ge30KCiAgICBmb3IgbW9kZSBpbiBNT0RFUzoKICAgICAgICBwcmludChmIlxueyc9JyAqIDY4fVxue21vZGV9ICAgW3NlZWQge3NlZWRfaSArIDF9XVxueyc9JyAqIDY4fSIsIGZsdXNoPVRydWUpCiAgICAgICAgciA9IEZseVJlc2Vydm9pcihjb25uZWN0b21lLCBlbWJlZGRpbmdfZGltPTI1NiwgZGltcz0xMjgsIHNlZWQ9cl9zZWVkKQogICAgICAgIHIuc2V0X21vZGUobW9kZSkKICAgICAgICBpZiBVU0VfR1BVOgogICAgICAgICAgICBpbmZvID0gci5lbmFibGVfZ3B1KCkKICAgICAgICAgICAgcHJpbnQoZiIgIGRldmljZTogIiArIChmIkdQVSB7aW5mb1snZGV2aWNlJ119IiBpZiBpbmZvLmdldCgiZW5hYmxlZCIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZWxzZSBmIkNQVSAoe2luZm8uZ2V0KCdyZWFzb24nKX0pIiksIGZsdXNoPVRydWUpCiAgICAgICAgYnJhaW4gPSBQbGFzdGljQnJhaW4ociwgbl9wb29scz00LCBwb29sX3NpemU9MjAwLCBzZWVkPWJfc2VlZCwgc3RlcHM9NikKICAgICAgICBicmFpbi5zZXRfbW9kZShtb2RlKQoKICAgICAgICBjdXJ2ZSA9IFt7ImVwb2NoIjogMCwgImFjY3VyYWN5IjogYnJhaW4uYWNjdXJhY3koWCwgeSksICJsb3NzIjogTm9uZX1dCiAgICAgICAgcHJpbnQoZiIgICAgZXBvY2ggICAwOiBhY2Mge2N1cnZlWzBdWydhY2N1cmFjeSddOjYuMSV9ICAgKHVudHJhaW5lZCkiLCBmbHVzaD1UcnVlKQogICAgICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgICAgIGZvciBlcCBpbiByYW5nZSgxLCBFUE9DSFMgKyAxKToKICAgICAgICAgICAgaWR4ID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKGVwKS5wZXJtdXRhdGlvbihsZW4oWCkpCiAgICAgICAgICAgIGxvc3NlcyA9IFticmFpbi5vYnNlcnZlKFhbaV0sIHlbaV0pWyJsb3NzIl0gZm9yIGkgaW4gaWR4XQogICAgICAgICAgICBhY2MgPSBicmFpbi5hY2N1cmFjeShYLCB5KQogICAgICAgICAgICBjdXJ2ZS5hcHBlbmQoeyJlcG9jaCI6IGVwLCAiYWNjdXJhY3kiOiBhY2MsICJsb3NzIjogZmxvYXQobnAubWVhbihsb3NzZXMpKX0pCiAgICAgICAgICAgIHByaW50KGYiICAgIGVwb2NoIHtlcDo+M306IGFjYyB7YWNjOjYuMSV9ICAgbG9zcyB7bnAubWVhbihsb3NzZXMpOi40Zn0gICAiCiAgICAgICAgICAgICAgICAgIGYie3RpbWUucGVyZl9jb3VudGVyKCkgLSB0MDo1LjBmfXMiLCBmbHVzaD1UcnVlKQoKICAgICAgICB0YWlsID0gW2NbImFjY3VyYWN5Il0gZm9yIGMgaW4gY3VydmVbLVRBSUw6XV0KICAgICAgICBzID0gci5wbGFzdGljX3NjYWxlCiAgICAgICAgY2twdCA9IFJVTlMgLyBmImNvbGFiX3N7c2VlZF9pfV97bW9kZX0ubnB6IgogICAgICAgIG1ldGEgPSBicmFpbi5zYXZlKGNrcHQpCgogICAgICAgICMgUmVjb21wdXRlIHRoZSByZXBvcnRlZCBudW1iZXIgZnJvbSB0aGUgc2F2ZWQgYXJ0aWZhY3QsIHRocm91Z2ggdGhlIHNlcnZpbmcgZ3VhcmRzLgogICAgICAgIHZyID0gRmx5UmVzZXJ2b2lyKGNvbm5lY3RvbWUsIGVtYmVkZGluZ19kaW09MjU2LCBkaW1zPTEyOCwgc2VlZD1yX3NlZWQpCiAgICAgICAgdnIuc2V0X21vZGUobW9kZSkKICAgICAgICB2ZXJpZnkgPSBQbGFzdGljQnJhaW4odnIsIG5fcG9vbHM9NCwgcG9vbF9zaXplPTIwMCwgc2VlZD1iX3NlZWQsIHN0ZXBzPTYpCiAgICAgICAgdmVyaWZ5LnNldF9tb2RlKG1vZGUpCiAgICAgICAgdmVyaWZ5LmxvYWQoY2twdCkKICAgICAgICByZWxvYWRlZCA9IHZlcmlmeS5hY2N1cmFjeShYLCB5KQogICAgICAgIGlmIGFicyhyZWxvYWRlZCAtIGN1cnZlWy0xXVsiYWNjdXJhY3kiXSkgPiAxZS02OgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KAogICAgICAgICAgICAgICAgZiJ7bW9kZX0gc2VlZCB7c2VlZF9pfTogY2hlY2twb2ludCByZXByb2R1Y2VzIHtyZWxvYWRlZDouNGZ9IGJ1dCB0aGUgY3VydmUgIgogICAgICAgICAgICAgICAgZiJzYXlzIHtjdXJ2ZVstMV1bJ2FjY3VyYWN5J106LjRmfTsgcmVmdXNpbmcgdG8gcmVjb3JkIGFuIHVucmVwcm9kdWNpYmxlICIKICAgICAgICAgICAgICAgICJudW1iZXIiCiAgICAgICAgICAgICkKCiAgICAgICAgcGVyX3NlZWRbc2VlZF9pXVttb2RlXSA9IHsKICAgICAgICAgICAgImN1cnZlIjogY3VydmUsCiAgICAgICAgICAgICJzY29yZSI6IGZsb2F0KHN0YXRpc3RpY3MubWVhbih0YWlsKSksCiAgICAgICAgICAgICJzY29yZV9zdGRfd2l0aGluIjogZmxvYXQoc3RhdGlzdGljcy5wc3RkZXYodGFpbCkpLAogICAgICAgICAgICAiZmluYWxfYWNjdXJhY3kiOiBjdXJ2ZVstMV1bImFjY3VyYWN5Il0sCiAgICAgICAgICAgICJzdGFydF9hY2N1cmFjeSI6IGN1cnZlWzBdWyJhY2N1cmFjeSJdLAogICAgICAgICAgICAicGxhc3RpY19lZGdlcyI6IGludChzLnNpemUpLAogICAgICAgICAgICAic2NhbGVfbWF4IjogZmxvYXQocy5tYXgoKSksCiAgICAgICAgICAgICJzZWNvbmRzIjogdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQwLAogICAgICAgICAgICAiY2hlY2twb2ludCI6IGNrcHQubmFtZSwKICAgICAgICAgICAgImNoZWNrcG9pbnRfcmVwcm9kdWNlc19maW5hbCI6IGZsb2F0KHJlbG9hZGVkKSwKICAgICAgICAgICAgImNoZWNrcG9pbnRfdXBkYXRlcyI6IGludChtZXRhWyJ1cGRhdGVzIl0pLAogICAgICAgIH0KICAgICAgICBwcmludChmIiAgLS0+IHN0YXJ0IHtjdXJ2ZVswXVsnYWNjdXJhY3knXTouMSV9ICBmaW5hbC1lcG9jaCAiCiAgICAgICAgICAgICAgZiJ7Y3VydmVbLTFdWydhY2N1cmFjeSddOi4xJX0gIGxhc3Qte1RBSUx9IG1lYW4gIgogICAgICAgICAgICAgIGYie3Blcl9zZWVkW3NlZWRfaV1bbW9kZV1bJ3Njb3JlJ106LjElfSAoc2QgIgogICAgICAgICAgICAgIGYie3Blcl9zZWVkW3NlZWRfaV1bbW9kZV1bJ3Njb3JlX3N0ZF93aXRoaW4nXTouMSV9KSAgc2NhbGVzIG1heCB7cy5tYXgoKTouMmZ9IgogICAgICAgICAgICAgIGYiICB7cGVyX3NlZWRbc2VlZF9pXVttb2RlXVsnc2Vjb25kcyddOi4wZn1zICAiCiAgICAgICAgICAgICAgZiJyZXByb2R1Y2VkIHtyZWxvYWRlZDouMSV9IiwgZmx1c2g9VHJ1ZSkKCgpkZWYgY29sKG1vZGU6IHN0ciwga2V5OiBzdHIpIC0+IGxpc3Q6CiAgICByZXR1cm4gW3Blcl9zZWVkW3NdW21vZGVdW2tleV0gZm9yIHMgaW4gcmFuZ2UoU0VFRFMpXQoKCmRlZiBtZWFuX3N0ZCh2YWxzOiBsaXN0KSAtPiB0dXBsZVtmbG9hdCwgZmxvYXRdOgogICAgcmV0dXJuIChzdGF0aXN0aWNzLm1lYW4odmFscyksCiAgICAgICAgICAgIHN0YXRpc3RpY3MucHN0ZGV2KHZhbHMpIGlmIGxlbih2YWxzKSA+IDEgZWxzZSAwLjApCgoKcHJpbnQoZiJcbnsnPScgKiA3NH1cbkFDUk9TUyB7U0VFRFN9IFNFRURTICAgKHNjb3JlID0gbWVhbiBvZiB0aGUgbGFzdCB7VEFJTH0gZXBvY2hzKSIKICAgICAgZiJcbnsnPScgKiA3NH0iLCBmbHVzaD1UcnVlKQpwcmludChmInsnYXJtJzo8MTR9IHsnbWVhbic6Pjh9IHsnc2QnOj43fSB7J21pbic6Pjd9IHsnbWF4Jzo+N30geydmaW5hbCBtZWFuJzo+MTF9IiwKICAgICAgZmx1c2g9VHJ1ZSkKcHJpbnQoIi0iICogNzQsIGZsdXNoPVRydWUpCnN1bW1hcnkgPSB7fQpmb3IgbW9kZSBpbiBNT0RFUzoKICAgIG0sIHNkID0gbWVhbl9zdGQoY29sKG1vZGUsICJzY29yZSIpKQogICAgZmUsIF8gPSBtZWFuX3N0ZChjb2wobW9kZSwgImZpbmFsX2FjY3VyYWN5IikpCiAgICBzdW1tYXJ5W21vZGVdID0geyJzY29yZV9tZWFuIjogbSwgInNjb3JlX3NkIjogc2QsICJmaW5hbF9lcG9jaF9tZWFuIjogZmV9CiAgICBwcmludChmInttb2RlOjwxNH0ge206PjcuMSV9IHtzZDo+Ni4xJX0ge21pbihjb2wobW9kZSwgJ3Njb3JlJykpOj42LjElfSAiCiAgICAgICAgICBmInttYXgoY29sKG1vZGUsICdzY29yZScpKTo+Ni4xJX0ge2ZlOj4xMC4xJX0iLCBmbHVzaD1UcnVlKQoKZ2Fwc19iZXN0LCBnYXBzX3NodWYsIGdhcHNfcmFuZCA9IFtdLCBbXSwgW10KcHJpbnQoIlxucGFpcmVkIHBlciBzZWVkIChzYW1lIHNlZWQsIHNhbWUgb3JkZXIsIG9ubHkgdGhlIHdpcmluZyBkaWZmZXJzKToiLCBmbHVzaD1UcnVlKQpwcmludChmInsnc2VlZCc6PjV9IHsnaW50YWN0Jzo+OH0geydzaHVmZmxlZCc6Pjl9IHsncmFuZG9tJzo+OH0geydiZXN0IGN0bCc6Pjl9ICIKICAgICAgZiJ7J2dhcCc6Pjd9IiwgZmx1c2g9VHJ1ZSkKZm9yIHMgaW4gcmFuZ2UoU0VFRFMpOgogICAgaSA9IHBlcl9zZWVkW3NdWyJpbnRhY3QiXVsic2NvcmUiXQogICAgc2ggPSBwZXJfc2VlZFtzXVsic2h1ZmZsZWQiXVsic2NvcmUiXQogICAgcmcgPSBwZXJfc2VlZFtzXVsicmFuZG9tX2dyYXBoIl1bInNjb3JlIl0KICAgIGJlc3QgPSBtYXgoc2gsIHJnKQogICAgZ2Fwc19iZXN0LmFwcGVuZChpIC0gYmVzdCkKICAgIGdhcHNfc2h1Zi5hcHBlbmQoaSAtIHNoKQogICAgZ2Fwc19yYW5kLmFwcGVuZChpIC0gcmcpCiAgICBwcmludChmIntzICsgMTo+NX0ge2k6PjcuMSV9IHtzaDo+OC4xJX0ge3JnOj43LjElfSB7YmVzdDo+OC4xJX0ge2kgLSBiZXN0Oj4rNi4xJX0iLAogICAgICAgICAgZmx1c2g9VHJ1ZSkKCmdtLCBnc2QgPSBtZWFuX3N0ZChnYXBzX2Jlc3QpCnNtLCBzc2QgPSBtZWFuX3N0ZChnYXBzX3NodWYpCnJtLCByc2QgPSBtZWFuX3N0ZChnYXBzX3JhbmQpCndpbnMgPSBzdW0oMSBmb3IgZyBpbiBnYXBzX2Jlc3QgaWYgZyA+IDApCnByaW50KGYiXG4gIGludGFjdCAtIGJlc3QgY29udHJvbCA6IHtnbTorLjElfSAgKHNkIHtnc2Q6LjElfSwgIgogICAgICBmInNlZWRzIHdoZXJlIGludGFjdCBsZWFkcyB7d2luc30ve1NFRURTfSkiLCBmbHVzaD1UcnVlKQpwcmludChmIiAgaW50YWN0IC0gc2h1ZmZsZWQgICAgIDoge3NtOisuMSV9ICAoc2Qge3NzZDouMSV9KSIsIGZsdXNoPVRydWUpCnByaW50KGYiICBpbnRhY3QgLSByYW5kb21fZ3JhcGggOiB7cm06Ky4xJX0gIChzZCB7cnNkOi4xJX0pIiwgZmx1c2g9VHJ1ZSkKCnByaW50KGYiXG57Jz0nICogNzR9XG5XSEFUIFRIRSBOVU1CRVJTIFNVUFBPUlRcbnsnPScgKiA3NH0iLCBmbHVzaD1UcnVlKQpwcmludChmIiAgbWFqb3JpdHktY2xhc3MgYmFzZWxpbmUgOiB7Y2hhbmNlOi4xJX0iLCBmbHVzaD1UcnVlKQpwcmludChmIiAgZWRnZS1mcmVlIGNvbnRyb2wgICAgICAgOiB7c3VtbWFyeVsnbm9fZWRnZXMnXVsnc2NvcmVfbWVhbiddOi4xJX0gIgogICAgICBmIihzdGF0ZSBpcyBleGFjdGx5IHplcm8sIHNvIHRoaXMgbXVzdCBzdGF5IGF0IGNoYW5jZSkiLCBmbHVzaD1UcnVlKQppZiBTRUVEUyA8IDM6CiAgICBwcmludCgiICBGZXdlciB0aGFuIDMgc2VlZHM6IHRvbyBmZXcgdG8gc2VwYXJhdGUgYW4gZWZmZWN0IGZyb20gc2VlZCBub2lzZS4iLAogICAgICAgICAgZmx1c2g9VHJ1ZSkKZWxpZiBhYnMoZ20pIDwgZ3NkIG9yIHdpbnMgaW4gKDAsIFNFRURTKToKICAgIHByaW50KGYiICBUaGUgaW50YWN0IGNvbm5lY3RvbWUncyB7Z206Ky4xJX0gb3ZlciB0aGUgYmVzdCBjb250cm9sIHNpdHMgaW5zaWRlIHRoZSAiCiAgICAgICAgICBmInNlZWQtdG8tc2VlZCBzcHJlYWQgKHtnc2Q6LjElfSkgYW5kIGRvZXMgbm90IGNsZWFyIGl0LiBObyB3aXJpbmcgYWR2YW50YWdlIGlzIiwKICAgICAgICAgIGZsdXNoPVRydWUpCiAgICBwcmludCgiICBkZXRlY3RhYmxlIG9uIHRoaXMgdGFzayB3aXRoIHRoaXMgZGVzaWduLiBBIHJlY3VycmVudCBncmFwaCBpcyBzdGlsbCByZXF1aXJlZDoiLAogICAgICAgICAgZmx1c2g9VHJ1ZSkKICAgIHByaW50KCIgIHRoZSBlZGdlLWZyZWUgY29udHJvbCBzdGF5cyBhdCBjaGFuY2Ugd2hpbGUgZXZlcnkgY29ubmVjdGVkIGFybSBsZWFybnMuIiwKICAgICAgICAgIGZsdXNoPVRydWUpCmVsc2U6CiAgICBwcmludChmIiAgVGhlIGludGFjdCBjb25uZWN0b21lIGxlYWRzIHRoZSBiZXN0IGNvbnRyb2wgYnkge2dtOisuMSV9IGFnYWluc3QgYSBzZWVkIgogICAgICAgICAgZiIgc3ByZWFkIG9mIHtnc2Q6LjElfSwgb24ge3dpbnN9L3tTRUVEU30gc2VlZHMuIFRoYXQgY2xlYXJzIHRoZSBub2lzZSBoZXJlLiIsCiAgICAgICAgICBmbHVzaD1UcnVlKQogICAgcHJpbnQoIiAgUmVwbGljYXRlIGJlZm9yZSBjYWxsaW5nIGl0IGEgcHJvcGVydHkgb2YgdGhlIGZseS4iLCBmbHVzaD1UcnVlKQoKcmVwb3J0ID0gewogICAgImNvbmZpZyI6IHsKICAgICAgICAiZXBvY2hzIjogRVBPQ0hTLCAic2VlZHMiOiBTRUVEUywgInRhaWwiOiBUQUlMLAogICAgICAgICMgRGVyaXZlZCwgbm90IGFzc2VydGVkLiBUaGlzIHdhcyBhIGhhcmRjb2RlZCBzdHJpbmcgYW5kIHRoZSBmaXJzdCBDUFUgcnVuIHdyb3RlCiAgICAgICAgIyAiR1BVIChjb2xhYikiIGludG8gaXRzIG93biBhcnRpZmFjdCwgd2hpY2ggaXMgdGhlIG1pc2xhYmVsbGVkLXByb3ZlbmFuY2UgZmFpbHVyZSB0aGUKICAgICAgICAjIHByb2plY3QgaGFzIGhhZCB0byBmaXggYmVmb3JlOiBhIG51bWJlciB3aG9zZSByZWNvcmRlZCBvcmlnaW4gaXMgd3JvbmcuCiAgICAgICAgImRldmljZSI6ICJHUFUiIGlmIFVTRV9HUFUgZWxzZSAiQ1BVIiwKICAgICAgICAic3RlcHMiOiA2LCAicG9vbF9zaXplIjogMjAwLCAibl9wb29scyI6IDQsCiAgICB9LAogICAgInBlcl9zZWVkIjoge3N0cihrKTogdiBmb3IgaywgdiBpbiBwZXJfc2VlZC5pdGVtcygpfSwKICAgICJzdW1tYXJ5Ijogc3VtbWFyeSwKICAgICJnYXAiOiB7InZzX2Jlc3RfY29udHJvbF9tZWFuIjogZ20sICJ2c19iZXN0X2NvbnRyb2xfc2QiOiBnc2QsCiAgICAgICAgICAgICJ2c19zaHVmZmxlZF9tZWFuIjogc20sICJ2c19zaHVmZmxlZF9zZCI6IHNzZCwKICAgICAgICAgICAgInZzX3JhbmRvbV9tZWFuIjogcm0sICJ2c19yYW5kb21fc2QiOiByc2QsCiAgICAgICAgICAgICJzZWVkc19pbnRhY3RfbGVhZHMiOiB3aW5zfSwKICAgICJjaGFuY2VfbWFqb3JpdHkiOiBjaGFuY2UsCn0Kb3V0ID0gUlVOUyAvICJjb2xhYl9tZWFzdXJlLmpzb24iCm91dC53cml0ZV90ZXh0KGpzb24uZHVtcHMocmVwb3J0LCBpbmRlbnQ9MiksIGVuY29kaW5nPSJ1dGYtOCIpCnByaW50KGYiXG53cm90ZSB7b3V0fSIsIGZsdXNoPVRydWUpCgp6aXBfcGF0aCA9IFJVTlMgLyAiY29sYWJfcmVzdWx0cy56aXAiCndpdGggemlwZmlsZS5aaXBGaWxlKHppcF9wYXRoLCAidyIsIHppcGZpbGUuWklQX0RFRkxBVEVEKSBhcyB6OgogICAgZm9yIGYgaW4gc29ydGVkKFJVTlMuZ2xvYigiY29sYWJfKiIpKToKICAgICAgICB6LndyaXRlKGYsIGYubmFtZSkKICAgIHJlcG9ydCA9IEdSQVBILnBhcmVudCAvICJzdGVwMV9yZXBvcnQuanNvbiIKICAgIGlmIHJlcG9ydC5leGlzdHMoKToKICAgICAgICB6LndyaXRlKHJlcG9ydCwgInN0ZXAxX3JlcG9ydC5qc29uIikKcHJpbnQoZiJ3cm90ZSB7emlwX3BhdGh9ICh7emlwX3BhdGguc3RhdCgpLnN0X3NpemUgLyAxZTY6LjFmfSBNQikiLCBmbHVzaD1UcnVlKQo=',
}

DIGESTS = {'payload.py': '81776ff70ec35e23228f5eadbe725f8e09ed460d2c2d08be3197d3aa35953bfa', 'step1_build.py': 'f576218d66fb93d0d0dfb481491bd12f79c87c2798d3689b3e27dcb6602beb57', 'step2_measure.py': 'c7b92eb28244caaa339754b0ed3cdff66b00be08f965d1d4ffd25791f8832f4d'}
CONTENT = Path("/content")
ROOT = CONTENT / "flylingo"

for name, blob in FILES.items():
    raw = base64.b64decode(blob)
    got = hashlib.sha256(raw).hexdigest()
    assert got == DIGESTS[name], f"{name} decoded to {got}, expected {DIGESTS[name]}"
    (CONTENT / name).write_bytes(raw)
    print(f"wrote {name}: {len(raw)} bytes, sha256 verified")

## 3. The data

The connectome is public. `step1_build.py` downloads two Feather files over HTTPS and refuses to
build anything unless they match their exact byte counts and SHA-256 — a truncated download would
otherwise produce a plausible-looking wrong graph.

The published counts are then asserted, and the build fails loudly if any disagrees:

- **166,700** neurons (annotated, non-glial)
- **25,582,938** directed edges between them
- **124,177,617** synaptic contacts summed over those edges

Nothing is uploaded, and Colab's disk is ephemeral, so this re-runs in about five minutes per
session. That is the cost of not needing an account or a dataset upload.

In [ ]:
t0 = time.time()
sys.argv = ["step1_build.py"]
exec(compile((CONTENT / "step1_build.py").read_text(), "step1_build.py", "exec"),
     {"__name__": "__main__"})
print(f"\nstep 1 took {(time.time() - t0) / 60:.1f} min")

### Why step 1 gates the accelerator before measuring anything

An accelerated path is a second implementation of the same quantity, so it has to be shown to agree
with the reference before its numbers mean anything. This project learned that the hard way: a
`cupyx` sparse matvec was canonicalising the uploaded control matrix **in place**, merging its
31,231 duplicate `(row, column)` pairs and renumbering every entry after each merge, while the
trainer wrote weights **by position**. The accelerator therefore trained a matrix the CPU never had,
which disagreed by 6.0e-01 on a state of scale 0.8 after only six plasticity steps — and agreed
perfectly on an untrained brain, so a naive test passed.

The gate checks three things per arm:

1. an untrained settle agrees with the CPU to a **stated** float32 tolerance,
2. the uploaded matrix keeps the CPU's exact layout (stored-entry count and index pointers), and
3. after training, the weights have landed on the **same entries** on both devices.

The remaining cross-device difference is reported, not gated: a training loop feeds each step's
summation-order difference back into the weights, so drift of order 1e-3 is expected and is not a
defect. Gating on it would fail a correct implementation.

## 4. The measurement

Rules this follows, each of which the project got wrong at least once before:

- **An arm's score is the mean over the last N epochs, never one epoch.** Single-epoch accuracy on a
  converged run was measured swinging with a standard deviation of 7–8.5 points, with a within-run
  range up to 36 points.
- **Several seeds, compared paired per seed**, so a seed's difficulty cancels and only the mechanism
  under test differs.
- **Every arm saves its trained weights**, and the number it reports is recomputed by loading that
  checkpoint back through the same guards the service uses.
- **No pre-written verdict.** The summary prints the effect next to its own spread and refuses to
  claim when the effect does not clear it.
- **The read-out is part of the experiment.** An earlier design read the populations with an equal-weight
  mean and concluded the wiring mattered; that spread belonged to the read-out, and the claim was
  retracted. Here each pool is read with learned weights, which is also what a real output neuron does.

Default budget: **25 epochs, 3 seeds**. Raise the first two numbers for a tighter estimate; on a T4,
40 epochs x 5 seeds takes about 45 minutes.

In [ ]:
EPOCHS, SEEDS, TAIL = 25, 3, 10   # modest by default so this notebook is runnable end to end
os.environ["FLYLINGO_ROOT"] = str(ROOT)
sys.argv = ["step2_measure.py", str(EPOCHS), str(SEEDS), str(TAIL)]
exec(compile((CONTENT / "step2_measure.py").read_text(), "step2_measure.py", "exec"),
     {"__name__": "__main__"})

## 5. How to read the result

The number that answers the question is **`intact − best control`**, read against its own spread and
the count of seeds where intact leads.

Two traps, both of which this project fell into:

- **Do not read a single epoch.** If the summary says the effect is within the spread, there is no
  detectable difference — that is the result, not a failure to find one.
- **`best control` is a `max()` over two noisy arms.** It is a useful summary, but it biases the
  comparison *against* `intact`, so "intact is 1.4 points behind the best control" is not evidence
  that the real wiring is worse. Read the two per-control differences (`− shuffled`, `− random_graph`)
  separately, each against its own spread.

Also worth checking yourself: the `no_edges` arm must sit at chance. If it does not, something is
wrong with the harness and no other number should be believed. Its state is exactly zero, so its
pools are all equal and its argmax is degenerate.

## 6. What this notebook found when it was run

Colab T4, 40 epochs, 5 seeds, score = mean of the last 10 epochs:

| arm | mean | sd |
|---|---|---|
| **intact** | **87.7%** | 1.8% |
| shuffled | 88.2% | 2.3% |
| random_graph | 87.0% | 1.9% |
| no_edges | 21.6% | 0.0% |

Per-control differences: **−0.5%** (sd 2.1) against the shuffle, **+0.7%** (sd 1.7) against the
random graph. Intact led the best control on **0 of 5** seeds.

**The specific wiring does not matter on this task.** What does: a recurrent graph is required —
the edge-free control sits at 21.6%, below the 26.8% majority-class baseline, while every connected
arm reaches 87–88%.

A local CPU run at 12 epochs and 3 seeds (63.7% / 62.8% / 64.3% / 21.6%) ties the same way, and all
20 T4 checkpoints were re-loaded on a different machine and their accuracies recomputed from
scratch: 20/20 reproduced the reported number exactly.

This is a **null result, and it is the third time this project reached one** — twice before it was
retracted, once because the read-out was the ceiling and once because the accelerator was corrupting
the weights. Those reasons are fixed and tested now, which is why the null is reported rather than
explained away.

## 7. License and citation

The connectome is the **MaleCNS v1.0** release from the FlyEM project at Janelia Research Campus
(Google/Janealia `flyem-male-cns` public bucket), and is used here unmodified apart from the
retention rule and row normalisation described above. Check its own license before redistributing
the data; this notebook only downloads it at runtime.